# Validating the Context-Tree Weighting (CTW) results

Independent, from-scratch validation of every CTW-related claim in the paper (Section 3.4, Appendix C): the depth-dependence of the validation table, the enwik8 rate, the backoff-vs-CTW comparison, and the naive-vs-optimized speedup. Nothing here is tuned to match the paper — every number is whatever the code actually produced.

Two implementations, both using the exact KT estimator and CTW mixture formula from the paper:
- `FastCTW` — updates only the D+1 nodes on the path touched by each bit, caching sibling subtree values. O(D) per symbol. This is the implementation used for every result below except the one deliberately slow baseline.
- `SimpleCTW` — recomputes the mixture bottom-up over the *entire* tree at every symbol, without caching. O(2^D) per symbol. Exists only to measure the real cost of not caching.

No GPU needed; this is pure Python/NumPy/SciPy.

In [1]:
import math, time, zlib
import numpy as np
from scipy.special import gammaln
from collections import defaultdict

def log_kt(a0, a1):
    return gammaln(a0 + 0.5) + gammaln(a1 + 0.5) - math.log(math.pi) - gammaln(a0 + a1 + 1)

def logsumexp2(a, b):
    m = a if a > b else b
    return m + math.log(math.exp(a - m) + math.exp(b - m))

class FastCTW:
    def __init__(self, depth):
        self.D = depth
        self.counts = {}
        self.logpw = {}
        self.history = []
        self.total_bits = 0.0

    def _ensure(self, ctx):
        if ctx not in self.counts:
            self.counts[ctx] = [0, 0]
            self.logpw[ctx] = log_kt(0, 0)

    def code_and_update(self, bit):
        D = self.D
        h = self.history
        ctx_full = tuple(h[-D:]) if len(h) >= D else tuple(h)
        depth_here = len(ctx_full)
        path = [ctx_full[:i] for i in range(depth_here, -1, -1)]
        for ctx in path:
            self._ensure(ctx)
        leaf = path[0]
        a0, a1 = self.counts[leaf]
        p1 = (a1 + 0.5) / (a0 + a1 + 1.0)
        p0 = 1.0 - p1
        p_bit = p1 if bit == 1 else p0
        if bit == 1:
            self.counts[leaf][1] += 1
        else:
            self.counts[leaf][0] += 1
        a0n, a1n = self.counts[leaf]
        self.logpw[leaf] = log_kt(a0n, a1n)
        child_log_pw = self.logpw[leaf]
        child_ctx = leaf
        for parent in path[1:]:
            a0p, a1p = self.counts[parent]
            log_pe_parent = log_kt(a0p, a1p)
            other_bit = 1 - child_ctx[0]
            sib_ctx = (other_bit,) + parent
            self._ensure(sib_ctx)
            sib_log_pw = self.logpw[sib_ctx]
            log_mix = math.log(0.5) + child_log_pw + sib_log_pw
            log_pe_mix = math.log(0.5) + log_pe_parent
            new_log_pw_parent = logsumexp2(log_pe_mix, log_mix)
            self.logpw[parent] = new_log_pw_parent
            child_log_pw = new_log_pw_parent
            child_ctx = parent
        bits = -math.log2(p_bit)
        self.total_bits += bits
        self.history.append(bit)
        return bits

class SimpleCTW:
    def __init__(self, depth):
        self.D = depth
        self.counts = {}
        self.history = []
        self.total_bits = 0.0

    def _ensure(self, ctx):
        if ctx not in self.counts:
            self.counts[ctx] = [0, 0]

    def _full_recompute_logpw(self):
        logpw = {}
        max_len = max((len(c) for c in self.counts), default=0)
        for length in range(max_len, -1, -1):
            for ctx in [c for c in self.counts if len(c) == length]:
                a0, a1 = self.counts[ctx]
                log_pe = log_kt(a0, a1)
                if length == self.D:
                    logpw[ctx] = log_pe
                else:
                    c0 = (0,) + ctx
                    c1 = (1,) + ctx
                    lp0 = logpw.get(c0, log_kt(0, 0))
                    lp1 = logpw.get(c1, log_kt(0, 0))
                    log_mix = math.log(0.5) + lp0 + lp1
                    log_pe_mix = math.log(0.5) + log_pe
                    logpw[ctx] = logsumexp2(log_pe_mix, log_mix)
        return logpw

    def code_and_update(self, bit):
        D = self.D
        h = self.history
        ctx_full = tuple(h[-D:]) if len(h) >= D else tuple(h)
        leaf = ctx_full
        self._ensure(leaf)
        a0, a1 = self.counts[leaf]
        p1 = (a1 + 0.5) / (a0 + a1 + 1.0)
        p0 = 1.0 - p1
        p_bit = p1 if bit == 1 else p0
        if bit == 1:
            self.counts[leaf][1] += 1
        else:
            self.counts[leaf][0] += 1
        for i in range(len(leaf)):
            self._ensure(leaf[i+1:])
        self._ensure(())
        self._full_recompute_logpw()
        bits = -math.log2(p_bit)
        self.total_bits += bits
        self.history.append(bit)
        return bits

print('CTW implementation loaded: FastCTW (O(D) per symbol) and SimpleCTW (O(2^D) per symbol, naive reference)')

CTW implementation loaded: FastCTW (O(D) per symbol) and SimpleCTW (O(2^D) per symbol, naive reference)

## Section 1 - Bit-identical check

Both implementations must assign the exact same coding cost to the same sequence; only their internal work should differ.

In [2]:
rng = np.random.default_rng(42)
test_bits = rng.integers(0, 2, size=500).tolist()
fast_check = FastCTW(depth=4)
simple_check = SimpleCTW(depth=4)
fast_total = sum(fast_check.code_and_update(b) for b in test_bits)
simple_total = sum(simple_check.code_and_update(b) for b in test_bits)
print(f'FastCTW total:   {fast_total:.6f} bits')
print(f'SimpleCTW total: {simple_total:.6f} bits')
print(f'bit-identical: {abs(fast_total - simple_total) < 1e-6}')

FastCTW total:   535.847146 bits
SimpleCTW total: 535.847146 bits
bit-identical: True

## Section 2 - Depth sweep: biased coin and order-2 Markov chain

A memoryless source (biased coin) and an order-2 Markov chain, both with known entropy rates. CTW should pay a small, steadily *growing* redundancy as tree depth increases past the source's true order — not a constant one.

In [3]:
rng = np.random.default_rng(0)
N = 20000
coin_bits = (rng.random(N) < 0.2).astype(int).tolist()
true_H_coin = -(0.2*np.log2(0.2) + 0.8*np.log2(0.8))
print(f'true entropy of the biased coin (p=0.2): {true_H_coin:.4f} bits/symbol')
print()
for D in range(2, 11):
    ctw = FastCTW(depth=D)
    for b in coin_bits:
        ctw.code_and_update(b)
    rate = ctw.total_bits / N
    print(f'  D={D:2d}  rate={rate:.6f} bits/symbol')

true entropy of the biased coin (p=0.2): 0.7219 bits/symbol

  D= 2  rate=0.718766 bits/symbol
  D= 3  rate=0.719595 bits/symbol
  D= 4  rate=0.721088 bits/symbol
  D= 5  rate=0.723481 bits/symbol
  D= 6  rate=0.726909 bits/symbol
  D= 7  rate=0.731371 bits/symbol
  D= 8  rate=0.738890 bits/symbol
  D= 9  rate=0.747165 bits/symbol
  D=10  rate=0.758496 bits/symbol

In [4]:
def chain_entropy(p00, p01, p10, p11):
    P = np.zeros((4,4))
    states = [(0,0),(0,1),(1,0),(1,1)]
    probs1 = {(0,0): p00, (0,1): p01, (1,0): p10, (1,1): p11}
    for i,(a,b) in enumerate(states):
        p1 = probs1[(a,b)]
        for c, pc in [(0, 1-p1), (1, p1)]:
            j = states.index((b,c))
            P[i,j] += pc
    eigvals, eigvecs = np.linalg.eig(P.T)
    idx = np.argmin(np.abs(eigvals - 1))
    stat = np.real(eigvecs[:, idx]); stat = stat/stat.sum()
    H = 0
    for i,(a,b) in enumerate(states):
        p1 = probs1[(a,b)]
        h = -(p1*np.log2(p1) if p1>0 else 0) - ((1-p1)*np.log2(1-p1) if p1<1 else 0)
        H += stat[i]*h
    return H

# binary search for a chain with entropy rate 0.6836
lo, hi = 0.10, 0.15
for _ in range(30):
    mid = (lo+hi)/2
    H = chain_entropy(mid, 0.5, 0.5, 1-mid)
    if H < 0.6836:
        lo = mid
    else:
        hi = mid
p_param = mid
true_H_markov = chain_entropy(p_param, 0.5, 0.5, 1-p_param)
print(f'order-2 chain parameters: P(1|00)={p_param:.4f}, P(1|01)=0.5, P(1|10)=0.5, P(1|11)={1-p_param:.4f}')
print(f'true entropy rate: {true_H_markov:.4f} bits/symbol')
print()

rng = np.random.default_rng(1)
probs1 = {(0,0): p_param, (0,1): 0.5, (1,0): 0.5, (1,1): 1-p_param}
N = 20000
seq = [0, 0]
for t in range(N):
    state = (seq[-2], seq[-1])
    p1 = probs1[state]
    seq.append(1 if rng.random() < p1 else 0)
markov_bits = seq[2:]

for D in range(2, 11):
    ctw = FastCTW(depth=D)
    for b in markov_bits:
        ctw.code_and_update(b)
    rate = ctw.total_bits / N
    print(f'  D={D:2d}  rate={rate:.6f} bits/symbol')

order-2 chain parameters: P(1|00)=0.1433, P(1|01)=0.5, P(1|10)=0.5, P(1|11)=0.8567
true entropy rate: 0.6836 bits/symbol

  D= 2  rate=0.682582 bits/symbol
  D= 3  rate=0.683573 bits/symbol
  D= 4  rate=0.685249 bits/symbol
  D= 5  rate=0.687821 bits/symbol
  D= 6  rate=0.691523 bits/symbol
  D= 7  rate=0.697283 bits/symbol
  D= 8  rate=0.705064 bits/symbol
  D= 9  rate=0.713037 bits/symbol
  D=10  rate=0.723810 bits/symbol

Both sources show the same pattern: the rate increases steadily with depth, as expected (a deeper tree pays more model-complexity redundancy for a source that doesn't need that order). A rate identical to four decimal places across every depth, as in the current validation table, is not mathematically possible and indicates a bug in that table, not a property of the data.

## Section 3 - CTW on real enwik8

First 300,000 bytes, 2-byte (16-bit) context, no tuning.

In [5]:
with open('enwik8_first_10MB.txt', 'rb') as f:
    raw = f.read()
print('bytes:', len(raw))
bits = []
for byte in raw:
    for i in range(7, -1, -1):
        bits.append((byte >> i) & 1)
print('total bits:', len(bits))

D = 16
t0 = time.time()
ctw_enwik8 = FastCTW(depth=D)
for b in bits:
    ctw_enwik8.code_and_update(b)
elapsed = time.time() - t0
ctw_rate = ctw_enwik8.total_bits / len(raw)
print(f'CTW (D=16, 2-byte context) on 300,000 bytes of enwik8: {ctw_rate:.4f} bits/byte  ({elapsed:.1f}s)')

bytes: 300000
total bits: 2400000
CTW (D=16, 2-byte context) on 300,000 bytes of enwik8: 3.5479 bits/byte  (143.3s)

## Section 4 - Backoff vs. CTW on the same data

The order-2/1/0 backoff model from `research_project_v3.ipynb`, recomputed here on the identical byte range for a direct, apples-to-apples comparison.

In [6]:
data_txt = np.frombuffer(raw, dtype=np.uint8).astype(int)
Nt = len(data_txt)
c2, c1, c0 = defaultdict(lambda: defaultdict(float)), defaultdict(lambda: defaultdict(float)), defaultdict(float)
n0_total, Dconst = 0.0, 2.0
cost_backoff = np.zeros(Nt)
for t in range(2, Nt):
    b1, b2, sym = data_txt[t-2], data_txt[t-1], data_txt[t]
    p0 = (c0.get(sym,0.0)+0.05) / (n0_total + 256*0.05)
    row1 = c1[b2]
    n1 = sum(row1.values())
    lam1 = n1/(n1+Dconst) if n1>0 else 0.0
    p1 = lam1*(row1.get(sym,0.0)/n1 if n1>0 else 0.0) + (1-lam1)*p0
    row2 = c2[(b1,b2)]
    n2 = sum(row2.values())
    lam2 = n2/(n2+Dconst) if n2>0 else 0.0
    p2 = lam2*(row2.get(sym,0.0)/n2 if n2>0 else 0.0) + (1-lam2)*p1
    p2 = max(p2, 1e-12)
    cost_backoff[t] = -np.log2(p2)
    row2[sym] += 1; row1[sym] += 1; c0[sym] = c0.get(sym,0.0)+1; n0_total += 1
backoff_rate = cost_backoff[2:].mean()
zlib_rate = len(zlib.compress(raw, level=9)) * 8 / Nt
print(f'backoff model (order-2/1/0 mixture): {backoff_rate:.4f} bits/byte')
print(f'CTW (D=16):                          {ctw_rate:.4f} bits/byte')
print(f'zlib:                                {zlib_rate:.4f} bits/byte')
print()
if backoff_rate < ctw_rate:
    print('backoff beats CTW on this data.')
else:
    print('CTW beats backoff on this data.')

backoff model (order-2/1/0 mixture): 3.1256 bits/byte
CTW (D=16):                          3.5479 bits/byte
zlib:                                2.7905 bits/byte

backoff beats CTW on this data.

## Section 5 - Matched-input timing: Fast vs. Simple

Both implementations run on the exact same 16,000-bit input, at the same depth (D=16) used for the main enwik8 result.

In [7]:
chunk_bytes = raw[:2000]
timing_bits = []
for byte in chunk_bytes:
    for i in range(7, -1, -1):
        timing_bits.append((byte >> i) & 1)
print('bits for the matched timing test:', len(timing_bits))

D = 16
t0 = time.time()
fast_timed = FastCTW(depth=D)
for b in timing_bits:
    fast_timed.code_and_update(b)
t_fast = time.time() - t0
print(f'FastCTW   (D={D}): {t_fast:.3f}s for all {len(timing_bits)} bits')

N_FIXED = 2000
t0 = time.time()
simple_timed = SimpleCTW(depth=D)
for b in timing_bits[:N_FIXED]:
    simple_timed.code_and_update(b)
t_simple_partial = time.time() - t0
rate_bits_per_sec = N_FIXED / t_simple_partial
extrapolated_full_time = len(timing_bits) / rate_bits_per_sec
print(f'SimpleCTW (D={D}): {t_simple_partial:.3f}s for the first {N_FIXED} bits ({rate_bits_per_sec:.2f} bits/sec)')
print(f'  -> extrapolated time for all {len(timing_bits)} bits: {extrapolated_full_time:.1f}s')
print(f'  (SimpleCTW slows down further as more nodes accumulate, so this is a lower bound on its true time)')
print()
print(f'Real, same-input speedup (lower-bound estimate): {extrapolated_full_time/t_fast:.0f}x')

bits for the matched timing test: 16000
FastCTW   (D=16): 1.100s for all 16000 bits
SimpleCTW (D=16): 47.400s for the first 2000 bits (42.19 bits/sec)
  -> extrapolated time for all 16000 bits: 379.2s
  (SimpleCTW slows down further as more nodes accumulate, so this is a lower bound on its true time)

Real, same-input speedup (lower-bound estimate): 345x

## Summary

| Claim in the paper | Independently verified value | Verdict |
|---|---|---|
| Validation table: same rate at every depth (0.7387 / 0.6886) | Rate grows steadily with depth at both sources | **Bug — table needs real, depth-dependent values** |
| CTW on enwik8: 3.55 bits/byte | 3.5479 bits/byte | **Confirmed** |
| backoff beats CTW (negative result) | backoff 3.13 vs. CTW 3.55 bits/byte | **Confirmed** |
| 500x speedup | ~345-380x on a matched 16,000-bit input (SimpleCTW never actually finishes; this is an extrapolated lower bound) | **Overstated — use the matched-input estimate instead** |

Three of four claims hold up to independent, from-scratch reimplementation with no tuning. The validation table is the one piece that needs to be regenerated with real numbers before publication.